In [7]:
import numpy as np
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

def smape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return 100 * np.mean(
        2 * np.abs(y_pred - y_true) /
        (np.abs(y_true) + np.abs(y_pred))
    )


def mase(y_true, y_pred, y_train):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_train = np.array(y_train)

    naive_error = np.mean(np.abs(np.diff(y_train)))
    return np.mean(np.abs(y_true - y_pred)) / (naive_error)

def theil_u(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    denominator = np.sqrt(np.mean(y_true ** 2) + np.mean(y_pred ** 2))
    u = rmse / denominator
    return u

# Loading the data
df = pd.read_excel('output_week.xlsx')

# Counting values and aggregating by week
data = df['WEEK'].value_counts().reset_index()
data.columns = ['WEEK', 'DATA']

# Converting week to a date representing the start of the week
data['WEEK_period'] = pd.to_datetime(data['WEEK'] + '-1', format='%G-%V-%u', errors='coerce')
data = data.sort_values('WEEK_period').reset_index(drop=True)

# ------------------------------
# Smoothing (5-week moving average)
# ------------------------------
data['DATA_smooth'] = data['DATA'].rolling(window=5, center=True).mean()
data['DATA_smooth'].fillna(method='bfill', inplace=True)
data['DATA_smooth'].fillna(method='ffill', inplace=True)

# Log transformation of the data
data['y'] = np.log1p(data[['DATA_smooth']])

/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1738/3591384744.py:45: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['DATA_smooth'].fillna(method='bfill', inplace=True)
/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1738/3591384744.py:45: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['DATA_smooth'].fillna(method='bfill', inplace=True)
/var/folders/36/lb3mdhtj2lldbybnz877q_hr0000gn/T/ipykernel_1738/359138474

In [9]:
# ------------------------------
# Time Series Cross Validation
# ------------------------------

tscv = TimeSeriesSplit(n_splits=10)

rmse_arima = []
mae_arima = []
mase_arima = []
smape_arima = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(data["y"]), 1):

    print(f"\nFold {fold}")

    # ------------------------------
    # Split temporal
    # ------------------------------

    y_train = data["y"].iloc[train_idx]
    y_test = data["y"].iloc[test_idx]

    # ------------------------------
    # Ajuste do ARIMA
    # ------------------------------

    model_arima = ARIMA(
        y_train,
        order=(4, 1, 5)
    )

    model_fit_arima = model_arima.fit()

    # ------------------------------
    # Forecast
    # ------------------------------

    y_pred = model_fit_arima.forecast(
        steps=len(y_test)
    )

    # ------------------------------
    # Voltar para a escala original
    # ------------------------------

    y_train_real = np.expm1(y_train.to_numpy())

    y_test_real = np.expm1(y_test.to_numpy())

    y_pred_real = np.expm1(np.asarray(y_pred))

    # ------------------------------
    # Métricas
    # ------------------------------

    rmse_value = np.sqrt(
        mean_squared_error(
            y_test_real,
            y_pred_real
        )
    )

    mae_value = mean_absolute_error(
        y_test_real,
        y_pred_real
    )

    mase_value = mase(
        y_test_real,
        y_pred_real,
        y_train_real
    )

    smape_value = smape(
        y_test_real,
        y_pred_real
    )

    # ------------------------------
    # Armazenar resultados
    # ------------------------------

    rmse_arima.append(rmse_value)
    mae_arima.append(mae_value)
    mase_arima.append(mase_value)
    smape_arima.append(smape_value)

    print(
        f"RMSE = {rmse_value:.4f} | "
        f"MAE = {mae_value:.4f} | "
        f"MASE = {mase_value:.4f} | "
        f"SMAPE = {smape_value:.4f}"
    )

# ------------------------------
# Salvar resultados
# ------------------------------

results_arima = pd.DataFrame({
    "Fold": range(1, len(rmse_arima) + 1),
    "RMSE": rmse_arima,
    "MAE": mae_arima,
    "MASE": mase_arima,
    "SMAPE": smape_arima
})

results_arima.to_csv(
    "ARIMA_results.csv",
    index=False
)


Fold 1
RMSE = 7.4259 | MAE = 6.8860 | MASE = 4.4610 | SMAPE = 35.7489

Fold 2


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood opt

RMSE = 3.7910 | MAE = 3.1246 | MASE = 1.9894 | SMAPE = 18.9269

Fold 3
RMSE = 5.5513 | MAE = 4.8652 | MASE = 3.3879 | SMAPE = 35.1379

Fold 4


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


RMSE = 2.1821 | MAE = 1.7534 | MASE = 1.2811 | SMAPE = 18.1644

Fold 5


...

/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


RMSE = 2.0381 | MAE = 1.6356 | MASE = 1.5265 | SMAPE = 14.9554

Fold 9


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


RMSE = 3.6757 | MAE = 3.2938 | MASE = 3.1904 | SMAPE = 34.4541

Fold 10
RMSE = 1.4562 | MAE = 1.0953 | MASE = 1.0781 | SMAPE = 16.6635


/Users/octaciliofsilvajunior/miniforge3/lib/python3.10/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
